# **FLAN-T5 Opinion Summarization**
---

Summarizing multiple opinions/comments using FLAN-T5-base model.

## Install Packages

In [1]:
import subprocess
import sys
import os

pkgs_to_install = [
    "accelerate",           # GPU Optimization
    "datasets",             # Data Loading
    "evaluate",             # Metrics Loading
    "rouge_score",          # ROUGE Metric
    "bert_score",           # BERTScore Metric (New!)
    "ftfy"                  # Text Cleaning (Optional but good)
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U"] + pkgs_to_install)
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="no")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["WANDB_DISABLED"] = "true"  # Hard disable W&B login

Configuration already exists at /root/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


## Fine Tuning
---



In [2]:
import os
import torch
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
import evaluate
import numpy as np
import nltk

os.environ["WANDB_DISABLED"] = "true" 
nltk.download("punkt")
nltk.download("punkt_tab")

MODEL_CHECKPOINT = "google/flan-t5-base"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
NUM_EPOCHS = 12

2026-01-22 16:11:32.377511: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769098292.400710    1125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769098292.407934    1125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769098292.425959    1125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769098292.425980    1125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769098292.425983    1125 computation_placer.cc:177] computation placer alr

In [3]:
# LOAD DATASET
TRAIN_PATH = "/kaggle/input/dataset-summ/dataset_train.csv"
TEST_PATH = "/kaggle/input/dataset-summ/dataset_test.csv"
VAL_PATH = "/kaggle/input/dataset-summ/dataset_val.csv"

data_files = {
    "train": TRAIN_PATH,
    "validation": VAL_PATH,
    "test": TEST_PATH
}

dataset = load_dataset("csv", data_files=data_files)
print(f"Loaded dataset: {dataset}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded dataset: DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 100
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 100
    })
})


In [4]:
# TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing data...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Tokenizing data...


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [5]:
# METRICS (ROUGE for Training Loop)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # --- FIX: Replace -100 in predictions with pad_token_id ---
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in labels (You already had this, keep it)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rest of your metric calculation...
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {key: value * 100 for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [6]:
import gc
import torch
import warnings
import os
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# --- CLEANUP PREVIOUS MEMORY ---
if 'model' in globals(): del model
torch.cuda.empty_cache()
gc.collect()

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
MODEL_CHECKPOINT = "google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# --- TRAINING ARGUMENTS ---
args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-thesis-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    
    # 2x GPUs allows us to keep this batch size (split 16 per GPU)
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=12,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=False,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

# --- TRAINER ---
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print(f"Using {torch.cuda.device_count()} GPUs for training.")
trainer.train()

# SAVE THE MODEL
import os
save_path = "/kaggle/working/final_thesis_model"
if not os.path.exists(save_path):
    os.makedirs(save_path)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved successfully to {save_path}")

Using 2 GPUs for training.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.149488,31.687700,9.310200,25.090800,25.098200,34.090000
2,No log,2.027083,33.479600,10.545300,26.471000,26.418500,35.260000
3,No log,1.959618,35.289400,11.720100,28.139600,28.136200,34.400000
4,2.387500,1.920650,34.749700,11.753300,27.598600,27.574300,34.650000
5,2.387500,1.888098,35.336700,12.268600,28.514700,28.514800,33.930000
6,2.387500,1.873873,34.885600,11.776600,28.047500,28.019600,34.110000
7,2.387500,1.861244,35.924600,12.246800,29.205100,29.192800,33.440000
8,2.006100,1.852343,35.489000,11.944100,28.228800,28.155800,33.460000
9,2.006100,1.846030,35.010800,11.788500,27.873700,27.847500,34.370000
10,2.006100,1.841277,35.412300,12.118700,28.325500,28.317900,33.770000


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Model saved successfully to /kaggle/working/final_thesis_model


## Evaluation
---

In [7]:
from transformers import AutoModelForSeq2SeqLM
import torch
import gc

# --- SETUP COMPARISON ---
print("Loading Base Model for side-by-side comparison...")
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to("cuda")

print("\n--- Sanity Check Inference (Comparison) ---")
test_sample = dataset["test"][0]["input_text"]
target_sample = dataset["test"][0]["target_text"]

print(f"Input:\n{test_sample}\n")

# Prepare inputs
inputs = tokenizer(test_sample, return_tensors="pt").input_ids.to("cuda")

# fine-tuned
outputs_ft = model.generate(inputs, max_new_tokens=100, do_sample=False)
prediction_ft = tokenizer.decode(outputs_ft[0], skip_special_tokens=True)

# base
outputs_base = base_model.generate(inputs, max_new_tokens=100, do_sample=False)
prediction_base = tokenizer.decode(outputs_base[0], skip_special_tokens=True)

print("-" * 50)
print(f"BASE MODEL Prediction:\n{prediction_base}")
print("-" * 50)
print(f"FINE-TUNED Prediction:\n{prediction_ft}")
print("-" * 50)
print(f"Actual Target:\n{target_sample}")
print("-" * 50)

del base_model
gc.collect()
torch.cuda.empty_cache()

Loading Base Model for side-by-side comparison...

--- Sanity Check Inference (Comparison) ---
Input:
Summarize the opinions of users who skeptical of/deny climate change.:
- Hey . How is this possible? Have you been lying about global warming? #LockHerUp #ClimateFraudster €¦
- MORONIC JILL STEIN SAYS Istanbul attack NOT Islam's fault...BUT everything to do with climate change
- The people saying "don't look at the sun" are the same people who say climate change is real — and we all know that's a lie.
- No genuine scientist actually says that there is man-made climate change
- Heh Big thaw coming like numerous years with as cold or colder starts, is that climate change? Is a Jan thaw,w…
- If liberals could invent the perfect problem to promote their policies, it would be climate change.

--------------------------------------------------
BASE MODEL Prediction:
The people who say "don't look at the sun" are the same people who say climate change is real — and we all know that's a lie.
-

In [8]:
# EVALUATION (ROUGE + BERTScore)
bertscore = evaluate.load("bertscore")
# Note: Reuse 'rouge' from above

# Helper function to generate summaries for the whole test set
def generate_summary_batch(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to("cuda") # Ensure we use the GPU

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH)

    batch["predicted_summary"] = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return batch

# Run generation on the Test Set
print("Generating summaries for test set... (This may take a minute)")
results = dataset["test"].map(generate_summary_batch, batched=True, batch_size=BATCH_SIZE)
predictions = results["predicted_summary"]
references = results["target_text"]

print("Calculating Scores...")
final_rouge = rouge.compute(predictions=predictions, references=references)
final_bert = bertscore.compute(predictions=predictions, references=references, lang="en")

print("\n" + "-"*30)
print("FINAL THESIS METRICS")
print("-"*30)
print(f"ROUGE-1: {final_rouge['rouge1'] * 100:.2f}")
print(f"ROUGE-2: {final_rouge['rouge2'] * 100:.2f}")
print(f"ROUGE-L: {final_rouge['rougeL'] * 100:.2f}")
print("-"*30)
print(f"BERTScore Precision: {np.mean(final_bert['precision']):.4f}")
print(f"BERTScore Recall:    {np.mean(final_bert['recall']):.4f}")
print(f"BERTScore F1:        {np.mean(final_bert['f1']):.4f}")
print("-"*30)

Generating summaries for test set... (This may take a minute)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Calculating Scores...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



------------------------------
FINAL THESIS METRICS
------------------------------
ROUGE-1: 33.53
ROUGE-2: 11.27
ROUGE-L: 26.74
------------------------------
BERTScore Precision: 0.9029
BERTScore Recall:    0.8996
BERTScore F1:        0.9012
------------------------------
